# repo-locate × FastContext — Colab integration test

This is an **executable integration test**, not a demo. It uses a public synthetic repository fixture with a real cross-file localization trap: the future-aware valuation kernel, the beam-ranking integration point, relevant tests, and several plausible decoys live in different files.

**No kaggri source code, commit SHA, replay, competition artifact, or private repository content is used.**

Choose **Runtime → Change runtime type → T4 GPU** (or better), then **Run all**. The benchmark runs FastContext three times and reports how often its final citations contain both the production integration point and the relevant planning test file.

In [ ]:
import platform, subprocess, sys
print('Notebook Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Keep vLLM isolated from Colab's preinstalled Torch/TorchAudio stack.
# Current vLLM guidance recommends uv with --torch-backend=auto so the
# Torch wheel matches the available NVIDIA driver/CUDA runtime.
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os, pathlib, shutil, subprocess
UV = '/root/.local/bin/uv'
VLLM_ENV = '/content/vllm-env'
VLLM = f'{VLLM_ENV}/bin/vllm'

subprocess.run([UV, 'python', 'install', '3.12'], check=True)
shutil.rmtree(VLLM_ENV, ignore_errors=True)
subprocess.run([UV, 'venv', '--python', '3.12', VLLM_ENV], check=True)
subprocess.run([UV, 'pip', 'install', '--python', f'{VLLM_ENV}/bin/python',
                'vllm', '--torch-backend=auto'], check=True)

# FastContext itself is also isolated as a uv tool.
subprocess.run([UV, 'tool', 'install', '--python', '3.12', '--force',
                'git+https://github.com/Alssndr0/fastcontext.git'], check=True)

shutil.rmtree('/content/repo-locate', ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/janposlusny/repo-locate.git',
                '/content/repo-locate'], check=True)

os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
assert shutil.which('fastcontext'), 'fastcontext CLI was not installed'
assert pathlib.Path(VLLM).exists(), 'isolated vLLM executable was not installed'
print('fastcontext:', shutil.which('fastcontext'))
print('vLLM:', VLLM)
subprocess.run([f'{VLLM_ENV}/bin/python', '-c',
                "import torch; print('isolated torch:', torch.__version__, 'CUDA:', torch.version.cuda)"],
               check=True)

In [ ]:
# Start the full-precision SFT checkpoint. float16 + 16k context is conservative for a T4.
# The qwen/fastcontext served name makes the client disable Qwen thinking and send top_k=20.
import pathlib, requests, subprocess, time

MODEL_ID = 'ShaunGves/FastContext-1.0-4B-SFT'
SERVED_NAME = 'qwen3-fastcontext-sft'
LOG = '/content/vllm-fastcontext.log'
log = open(LOG, 'w')
server = subprocess.Popen([
    VLLM, 'serve', MODEL_ID,
    '--served-model-name', SERVED_NAME,
    '--dtype', 'float16',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.95',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
    '--enable-prefix-caching',
    '--enforce-eager',
    '--host', '127.0.0.1', '--port', '8000',
], stdout=log, stderr=subprocess.STDOUT)
print('vLLM PID:', server.pid)

for _ in range(120):
    if server.poll() is not None:
        log.flush()
        print(pathlib.Path(LOG).read_text()[-12000:])
        raise RuntimeError('vLLM exited before becoming ready')
    try:
        response = requests.get('http://127.0.0.1:8000/v1/models', timeout=2)
        if response.ok:
            print('FastContext endpoint ready:', response.json()['data'][0]['id'])
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    log.flush()
    print(pathlib.Path(LOG).read_text()[-12000:])
    raise TimeoutError('vLLM did not become ready')

In [ ]:
# Run the real repo-locate wrapper + real FastContext CLI/model against the synthetic fixture.
import os, subprocess
env = os.environ.copy()
env.update({
    'PATH': '/root/.local/bin:' + env['PATH'],
    'BASE_URL': 'http://127.0.0.1:8000/v1',
    'MODEL': 'qwen3-fastcontext-sft',
    'API_KEY': 'local',
    'FASTCONTEXT_MAX_TURNS': '8',
    'FASTCONTEXT_MAX_TOKENS': '4000',
    'FASTCONTEXT_TRAJ_DIR': '/content/fastcontext-trajectories',
})
result = subprocess.run(
    ['python', '/content/repo-locate/tests/colab/run_benchmark.py', '--runs', '3',
     '--fixture-dir', '/content/beam-ranking-fixture'],
    env=env, text=True,
)
if result.returncode not in (0, 1):
    raise RuntimeError(f'benchmark runner failed with exit code {result.returncode}')

## Interpretation

A pass shows that the **real FastContext CLI + real model + repo-locate wrapper** can recover the intended production/test neighborhood from a cold repository under a controlled public benchmark.

This does **not** yet test whether Codex, Claude Code, or Agy autonomously decide to invoke the skill at the right time; that is a separate outer-agent policy test. A private historical benchmark can be run later without publishing private competition material.